# ADMIN · Structure classification baseline

서열 + 3D 구조에서 만든 **31개 기본 특징 → Logistic Regression**. CPU로 실행합니다. starter_kit.zip을 풀고 이 노트북을 같은 폴더에서 여세요. 저장소에서는 notebooks/에서 실행해도 됩니다.

표준화는 학습 부분에만 fit합니다. H-group 검증 분할을 사용하고, 시험 정답은 읽지 않습니다.

In [ ]:
from pathlib import Path
import sys
DATA = Path(".") if Path("train.csv").exists() else Path("../data/public")
if not Path("baseline.py").exists():
    sys.path.insert(0, str(Path("../scripts").resolve()))
from baseline import load_inputs
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from threadpoolctl import threadpool_limits
import pandas as pd
train, test, X, X_test, fold = load_inputs(DATA)
print(f"Train: {len(train):,} / Test: {len(test):,} / Features: {X.shape[1]}")
X.head()

## 1. 검증
`validation_split.csv`는 학습 샘플만 포함합니다. 동일 H-superfamily가 fit/validation 양쪽에 섞이지 않습니다. 이 점수는 숨김 시험 점수의 보장이 아닙니다.

In [ ]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0, class_weight="balanced", random_state=42))
with threadpool_limits(limits=2):
    model.fit(X.iloc[fold == "fit"], train.label[fold == "fit"])
    prediction = model.predict(X.iloc[fold == "validation"])
score = f1_score(train.label[fold == "validation"], prediction, labels=sorted(train.label.unique()), average="macro", zero_division=0)
print(f"Validation Macro F1: {score:.5f}")

## 2. 전체 학습 후 제출 파일 만들기
시험 데이터로 모델이나 전처리를 fit하지 않습니다. `id,label` 두 열만 제출합니다.

In [ ]:
with threadpool_limits(limits=2):
    model.fit(X, train.label)
    prediction = model.predict(X_test)
submission = pd.DataFrame({"id": test.id, "label": prediction})
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv", submission.shape)
submission.head()

## 3. 다음 실험
특징 구성과 정규화, 다른 분류기를 비교해 보세요. 제공된 서열과 좌표를 활용하는 방법은 자유롭게 선택할 수 있습니다.

`structure-format.md`에 파일 구성과 NumPy 로딩 방법이 있습니다. 좌표 누락 mask와 연속 구간 정보를 확인하세요.

출처: CATH-Plus v4.4 S40 · Orengo Group / UCL · CC BY 4.0. 원본 정답 검색 및 수동 정답 작성 금지.